In [3]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import pandas as pd
import numpy as np

from sdp.data import taxonomy as tax
from sdp.data.labeling import tier1
from sdp.data import splitting as sp
from sdp.data import sampling as smp

ROOT    = Path.cwd().parent
RAW     = ROOT / "data" / "raw" / "codenet_cpp_metadata.parquet"
INTERIM = ROOT / "data" / "interim"
REPORTS = ROOT / "reports"
FIG     = ROOT / "reports" / "figures"

In [2]:
status = pd.read_parquet(RAW, columns=["status"])["status"]
print(f"Rows: {len(status):,}")

# Raises UnknownVerdictError, naming every unrecognised verdict at once.
tier1.validate_vocabulary(status.unique())
print("Vocabulary validated: all verdicts documented.")

counts = status.value_counts()
discards = pd.DataFrame({
    "verdict": [v for v in counts.index if v in tier1.DISCARDED_VERDICTS],
})
discards["reason"] = discards["verdict"].map(tier1.VERDICT_TO_DISCARD_REASON).astype(str)
discards["count"]  = discards["verdict"].map(counts)
discards["pct_of_cpp"] = (discards["count"] / len(status) * 100).round(4)
discards = discards.sort_values("count", ascending=False).reset_index(drop=True)

n_kept = int(counts[list(tier1.KEPT_VERDICTS)].sum())
print(discards.to_string(index=False))
print(f"\nKept:      {n_kept:,}  ({n_kept/len(status)*100:.2f}%)")
print(f"Discarded: {len(status)-n_kept:,}  ({(len(status)-n_kept)/len(status)*100:.2f}%)")

del status
gc.collect()

Rows: 8,008,527
Vocabulary validated: all verdicts documented.
               verdict                reason  count  pct_of_cpp
   Time Limit Exceeded EFFICIENCY_CONSTRAINT 326340      4.0749
WA: Presentation Error         OUTPUT_FORMAT  26449      0.3303
 Memory Limit Exceeded EFFICIENCY_CONSTRAINT  14637      0.1828
 Output Limit Exceeded EFFICIENCY_CONSTRAINT    778      0.0097
   Judge Not Available  JUDGE_INFRASTRUCTURE     94      0.0012
  Query Limit Exceeded EFFICIENCY_CONSTRAINT     88      0.0011
        Internal error  JUDGE_INFRASTRUCTURE     78       0.001
    Judge System Error  JUDGE_INFRASTRUCTURE      7      0.0001

Kept:      7,640,056  (95.40%)
Discarded: 368,471  (4.60%)


0

In [3]:
ext = pd.read_parquet(RAW, columns=["filename_ext", "language"])
print("filename_ext values:")
print(ext["filename_ext"].value_counts().to_string())
print("\nlanguage values:")
print(ext["language"].value_counts().to_string())

del ext
gc.collect()

filename_ext values:
filename_ext
cpp    8008527

language values:
language
C++    8008527


0

In [4]:
COLUMNS = [
    "submission_id",       # primary key, part of the file path
    "problem_id",          # the split key
    "user_id",             # identifies resubmission chains
    "status",              # provenance for the label
    "date",                # temporal reporting
    "code_size",           # truncation analysis; cross-check on extraction
    "original_language",   # Tier 2 depends on this (GCC 5.4.1 vs 9.2.1 vs Clang)
]

df = pd.read_parquet(
    RAW,
    columns=COLUMNS,
    filters=[("status", "in", sorted(tier1.KEPT_VERDICTS))],
)
print(f"Rows read: {len(df):,}")

df["coarse_label"] = pd.Categorical(
    df["status"].map(tier1.VERDICT_TO_CLASS).astype(str),
    categories=[c.value for c in tax.COARSE_ORDER],
    ordered=True,
)

for col in ["problem_id", "user_id", "status", "original_language"]:
    df[col] = df[col].astype("category")
df["submission_id"] = df["submission_id"].astype("string[pyarrow]")
df["code_size"] = df["code_size"].astype("int32")

print(f"Memory: {df.memory_usage(deep=True).sum()/1e9:.2f} GB")
df.head(3)

Rows read: 7,640,056
Memory: 0.32 GB


,submission_id,problem_id,user_id,status,date,code_size,original_language,coarse_label
0,s317469200,p00000,u972675635,Compile Error,1530881659,250,C++,COMPILE_ERROR
1,s667847559,p00000,u972675635,Accepted,1530881789,250,C++11,ERROR_FREE
2,s160425098,p00000,u642752018,Accepted,1530897493,232,C++,ERROR_FREE


In [5]:
EXPECTED = {
    "ERROR_FREE":    4_353_049,
    "LOGICAL":       2_571_284,
    "COMPILE_ERROR":   376_053,
    "RUNTIME_ERROR":   339_670,
}

assert len(df) == 7_640_056, f"row count mismatch: {len(df):,}"
assert df["coarse_label"].isna().sum() == 0, "unlabelled rows present"
assert not df["submission_id"].duplicated().any(), "duplicate submission_id"

actual = df["coarse_label"].value_counts().sort_index()
print(actual.to_string())
assert {k: int(v) for k, v in actual.items()} == EXPECTED, "label distribution mismatch"

print(f"\nProblems: {df['problem_id'].nunique():,}")
print(f"Users:    {df['user_id'].nunique():,}")
print("\nAll checks passed.")

coarse_label
ERROR_FREE       4353049
COMPILE_ERROR     376053
RUNTIME_ERROR     339670
LOGICAL          2571284

Problems: 4,032
Users:    102,527

All checks passed.


In [6]:
MANIFEST = INTERIM / "labeled_manifest.parquet"

df.to_parquet(MANIFEST, compression="zstd", index=False)
discards.to_csv(REPORTS / "discard_summary.csv", index=False)

print(f"{MANIFEST.name}: {MANIFEST.stat().st_size/1e6:.1f} MB")
print(f"discard_summary.csv written ({len(discards)} rows)")

labeled_manifest.parquet: 92.0 MB
discard_summary.csv written (8 rows)


In [7]:
from sdp.data import splitting as sp

df = pd.read_parquet(INTERIM / "labeled_manifest.parquet")
print(f"Rows: {len(df):,}   problem_id dtype: {df['problem_id'].dtype}")

df["split"]        = sp.problem_level_split(df["problem_id"], seed=sp.DEFAULT_SEED)
df["random_split"] = sp.submission_level_split(df.index,      seed=sp.DEFAULT_SEED)

print(f"Seed: {sp.DEFAULT_SEED}   ratios: "
      f"{ {k.value: v for k, v in sp.DEFAULT_RATIOS.items()} }")
df[["submission_id", "problem_id", "coarse_label", "split", "random_split"]].head(3)

Rows: 7,640,056   problem_id dtype: category
Seed: 42   ratios: {'train': 0.6, 'val': 0.2, 'test': 0.2}


,submission_id,problem_id,coarse_label,split,random_split
0,s317469200,p00000,COMPILE_ERROR,val,val
1,s667847559,p00000,ERROR_FREE,val,train
2,s160425098,p00000,ERROR_FREE,val,train


In [10]:
import importlib
from sdp.data import splitting as sp
importlib.reload(sp)

<module 'sdp.data.splitting' from 'D:\\Dev\\Github\\transformer-defect-prediction\\src\\sdp\\data\\splitting.py'>

In [11]:
sp.assert_problem_disjoint(df["problem_id"], df["split"])
print("Disjointness: PASSED\n")

print("Achieved proportions and class counts")
print(sp.split_summary(df["split"], df["coarse_label"]).to_string())

print("\nProblems per split")
print(df.groupby("split", observed=True)["problem_id"].nunique().to_string())

print("\nClass composition within each split (% of split)")
ct = pd.crosstab(df["split"], df["coarse_label"], normalize="index") * 100
print(ct.round(2).to_string())

print("\nCorpus-wide reference (%)")
print((df["coarse_label"].value_counts(normalize=True).sort_index() * 100)
      .round(2).to_string())

Disjointness: PASSED

Achieved proportions and class counts
          rows   pct  ERROR_FREE  COMPILE_ERROR  RUNTIME_ERROR  LOGICAL
split                                                                  
train  4584034  60.0     2571993         228192         199961  1583888
val    1528011  20.0      896626          76498          67385   487502
test   1528011  20.0      884430          71363          72324   499894

Problems per split
split
train    1528
val      1252
test     1252

Class composition within each split (% of split)
coarse_label  ERROR_FREE  COMPILE_ERROR  RUNTIME_ERROR  LOGICAL
split                                                          
train              56.11           4.98           4.36    34.55
val                58.68           5.01           4.41    31.90
test               57.88           4.67           4.73    32.72

Corpus-wide reference (%)
coarse_label
ERROR_FREE       56.98
COMPILE_ERROR     4.92
RUNTIME_ERROR     4.45
LOGICAL          33.66


In [12]:
SPLITS = ROOT / "data" / "processed" / "splits"
SPLITS.mkdir(parents=True, exist_ok=True)

SPLIT_MANIFEST = SPLITS / "split_manifest.parquet"
df.to_parquet(SPLIT_MANIFEST, compression="zstd", index=False)

print(f"{SPLIT_MANIFEST.name}: {SPLIT_MANIFEST.stat().st_size/1e6:.1f} MB")
print(f"Columns: {list(df.columns)}")

split_manifest.parquet: 93.5 MB
Columns: ['submission_id', 'problem_id', 'user_id', 'status', 'date', 'code_size', 'original_language', 'coarse_label', 'split', 'random_split']


In [4]:
from sdp.data import sampling as smp

df = pd.read_parquet(ROOT / "data" / "processed" / "splits" / "split_manifest.parquet")
print(f"Manifest: {len(df):,} rows")

quotas = smp.resolve_quotas()
sample, report = smp.draw_sample(df, quotas=quotas, seed=smp.DEFAULT_SEED)

print(f"Sampled: {len(sample):,} rows   (seed {smp.DEFAULT_SEED})\n")
print(report.to_string(index=False))

Manifest: 7,640,056 rows
Sampled: 75,000 rows   (seed 42)

split  coarse_label  requested  available  taken  shortfall  problems  max_per_problem
train    ERROR_FREE       6000    2571993   6000          0      1523                4
train COMPILE_ERROR      18000     228192  18000          0      1314               18
train RUNTIME_ERROR      15000     199961  15000          0      1260               16
train       LOGICAL       6000    1583888   6000          0      1412                5
  val    ERROR_FREE       2000     896626   2000          0      1248                2
  val COMPILE_ERROR       6000      76498   6000          0      1045                7
  val RUNTIME_ERROR       5000      67385   5000          0      1006                6
  val       LOGICAL       2000     487502   2000          0      1137                2
 test    ERROR_FREE       2000     884430   2000          0      1246                2
 test COMPILE_ERROR       6000      71363   6000          0      1048  

In [5]:
assert report["shortfall"].sum() == 0, "supply shortfall — investigate before extracting"
assert len(sample) == 75_000, f"expected 75,000; got {len(sample):,}"
assert not sample["submission_id"].duplicated().any()

n_problems = sample["problem_id"].astype(str).nunique()
print(f"Distinct problems in the working corpus: {n_problems:,} of 4,032 "
      f"({n_problems/4032*100:.1f}%)")

print("\nRows per problem across the whole corpus")
per_problem = sample["problem_id"].astype(str).value_counts()
print(per_problem.describe(percentiles=[.5, .9, .99]).round(1).to_string())

print("\nCross-check: split x class counts")
print(pd.crosstab(sample["split"], sample["coarse_label"]).to_string())

sp.assert_problem_disjoint(sample["problem_id"], sample["split"])
print("\nDisjointness preserved in the sample: PASSED")

Distinct problems in the working corpus: 4,032 of 4,032 (100.0%)

Rows per problem across the whole corpus
count    4032.0
mean       18.6
std        13.0
min         1.0
50%        16.0
90%        42.0
99%        43.0
max        43.0

Cross-check: split x class counts
coarse_label  ERROR_FREE  COMPILE_ERROR  RUNTIME_ERROR  LOGICAL
split                                                          
train               6000          18000          15000     6000
val                 2000           6000           5000     2000
test                2000           6000           5000     2000

Disjointness preserved in the sample: PASSED


In [6]:
sample = sample.copy()
sample["archive_path"] = smp.archive_paths(sample)

SAMPLE_MANIFEST = INTERIM / "sample_manifest.parquet"
WANTED_PATHS    = INTERIM / "wanted_paths.txt"

sample.to_parquet(SAMPLE_MANIFEST, compression="zstd", index=False)
WANTED_PATHS.write_text("\n".join(sample["archive_path"]) + "\n", encoding="utf-8")

report.to_csv(REPORTS / "sampling_report.csv", index=False)

print(f"{SAMPLE_MANIFEST.name}: {SAMPLE_MANIFEST.stat().st_size/1e6:.2f} MB")
print(f"{WANTED_PATHS.name}: {WANTED_PATHS.stat().st_size/1e6:.2f} MB, "
      f"{len(sample):,} paths")
print(f"\nFirst three:")
print("\n".join(sample['archive_path'].head(3)))

sample_manifest.parquet: 2.27 MB
wanted_paths.txt: 3.60 MB, 75,000 paths

First three:
Project_CodeNet/data/p00000/C++/s582427538.cpp
Project_CodeNet/data/p00000/C++/s773824563.cpp
Project_CodeNet/data/p00000/C++/s110287235.cpp


In [7]:
SOURCES = ROOT / "data" / "processed" / "sources"

extracted_files = list(SOURCES.rglob("*.cpp"))
print(f"Extracted .cpp files on disk: {len(extracted_files):,}")
assert len(extracted_files) == 75_000, "count mismatch — investigate before proceeding"

# Every path should resolve to a row in sample_manifest via archive_path
sample = pd.read_parquet(INTERIM / "sample_manifest.parquet")
expected_rel_paths = set(
    sample["archive_path"].str.replace("Project_CodeNet/data/", "", regex=False)
)
actual_rel_paths = {
    str(p.relative_to(SOURCES)).replace("\\", "/") for p in extracted_files
}

missing = expected_rel_paths - actual_rel_paths
extra = actual_rel_paths - expected_rel_paths
print(f"Missing (expected but not on disk): {len(missing)}")
print(f"Extra (on disk but not expected):   {len(extra)}")
assert not missing and not extra, "extraction does not match the sample manifest exactly"
print("\nStructure check: PASSED")

Extracted .cpp files on disk: 75,000
Missing (expected but not on disk): 0
Extra (on disk but not expected):   0

Structure check: PASSED


In [8]:
disk_sizes = pd.Series(
    {str(p.relative_to(SOURCES)).replace("\\", "/"): p.stat().st_size
     for p in extracted_files},
    name="disk_bytes",
)

sample["rel_path"] = sample["archive_path"].str.replace("Project_CodeNet/data/", "", regex=False)
check = sample.set_index("rel_path")[["submission_id", "code_size"]].join(disk_sizes)

check["diff"] = check["disk_bytes"] - check["code_size"]
mismatches = check[check["diff"] != 0]

print(f"Rows compared: {len(check):,}")
print(f"Exact byte match: {(check['diff'] == 0).sum():,} "
      f"({(check['diff'] == 0).mean()*100:.2f}%)")
print(f"Mismatched:        {len(mismatches):,}")

if len(mismatches):
    print("\nDiff distribution for mismatches:")
    print(mismatches["diff"].describe().round(1).to_string())
    print("\nWorst 5:")
    print(mismatches.reindex(mismatches["diff"].abs().sort_values(ascending=False).index)
          .head(5).to_string())

Rows compared: 75,000
Exact byte match: 75,000 (100.00%)
Mismatched:        0


In [10]:
import random

random.seed(42)
sample_check = extracted_files if len(extracted_files) <= 2000 else \
    random.sample(extracted_files, 2000)

results = []
for p in sample_check:
    raw = p.read_bytes()
    try:
        raw.decode("utf-8")
        status = "utf-8-clean"
    except UnicodeDecodeError:
        try:
            raw.decode("shift_jis")
            status = "shift_jis-decodable"
        except UnicodeDecodeError:
            status = "undecodable (neither utf-8 nor shift_jis)"
    has_high_byte = any(b >= 0x80 for b in raw)
    results.append({"path": p, "status": status, "has_high_byte": has_high_byte})

enc_df = pd.DataFrame(results)
print(f"Encoding survey ({len(sample_check):,} files sampled, seed 42):\n")
print(enc_df["status"].value_counts().to_string())

n_high_byte = enc_df["has_high_byte"].sum()
print(f"\nFiles containing any byte >= 0x80: {n_high_byte:,} "
      f"({n_high_byte/len(enc_df)*100:.2f}%)")

problem = enc_df[enc_df["status"] != "utf-8-clean"]
if len(problem):
    print(f"\nNon-UTF-8 examples:")
    print(problem.head(5).to_string())

Encoding survey (2,000 files sampled, seed 42):

status
utf-8-clean    2000

Files containing any byte >= 0x80: 255 (12.75%)


In [13]:
import time

def hash_with_progress(paths, root):
    rows = []
    start = time.monotonic()
    for i, p in enumerate(paths, 1):
        exists = p.exists()
        rel = str(p.relative_to(root))
        if exists:
            rows.append({"path": rel, "sha256": dd.hash_file(p),
                         "bytes": p.stat().st_size, "exists": True})
        else:
            rows.append({"path": rel, "sha256": None, "bytes": None, "exists": False})
        if i % 2000 == 0:
            elapsed = time.monotonic() - start
            print(f"  [{elapsed:6.1f}s] {i:,}/{len(paths):,} "
                  f"({i/elapsed:.0f} files/sec)")
    return pd.DataFrame(rows)

hashes = hash_with_progress(file_paths, SOURCES)
print(f"Done. {hashes['exists'].sum():,} hashed.")

  [   0.7s] 2,000/75,000 (2725 files/sec)
  [   1.5s] 4,000/75,000 (2753 files/sec)
  [   2.2s] 6,000/75,000 (2762 files/sec)
  [   2.9s] 8,000/75,000 (2767 files/sec)
  [   3.6s] 10,000/75,000 (2782 files/sec)
  [   4.3s] 12,000/75,000 (2782 files/sec)
  [   5.0s] 14,000/75,000 (2774 files/sec)
  [   5.8s] 16,000/75,000 (2760 files/sec)
  [   6.6s] 18,000/75,000 (2730 files/sec)
  [   7.3s] 20,000/75,000 (2729 files/sec)
  [   8.0s] 22,000/75,000 (2739 files/sec)
  [   8.8s] 24,000/75,000 (2738 files/sec)
  [   9.5s] 26,000/75,000 (2741 files/sec)
  [  10.2s] 28,000/75,000 (2744 files/sec)
  [  10.9s] 30,000/75,000 (2751 files/sec)
  [  11.7s] 32,000/75,000 (2745 files/sec)
  [  12.4s] 34,000/75,000 (2747 files/sec)
  [  13.1s] 36,000/75,000 (2749 files/sec)
  [  13.8s] 38,000/75,000 (2751 files/sec)
  [  14.5s] 40,000/75,000 (2750 files/sec)
  [  15.2s] 42,000/75,000 (2757 files/sec)
  [  16.0s] 44,000/75,000 (2758 files/sec)
  [  16.6s] 46,000/75,000 (2764 files/sec)
  [  17.3s] 48,

In [14]:
report = dd.dedup_report(hashes)
for k, v in report.items():
    print(f"{k:30} {v}")

groups = dd.duplicate_groups(hashes)
print(f"\nDuplicate groups found: {len(groups)}")
if len(groups):
    print(groups.head(10).to_string())

files_hashed                   75000
files_missing                  0
unique_hashes                  73615
duplicate_groups               1070
files_in_duplicate_groups      2455
redundant_files                1385
duplicate_rate_pct             1.8467

Duplicate groups found: 1070
                                                             sha256  count                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   

In [16]:
hashes_indexed = hashes.set_index("path")
hashes_indexed.index = hashes_indexed.index.str.replace("\\", "/", regex=False)

sample_rel = sample.copy()
sample_rel["rel_path"] = sample_rel["archive_path"].str.replace(
    "Project_CodeNet/data/", "", regex=False
)

joined = sample_rel.join(hashes_indexed[["sha256"]], on="rel_path")
assert joined["sha256"].notna().all(), "some sampled files failed to hash — investigate"

overlap = dd.cross_split_duplicate_hashes(joined)
print("Cross-split content-hash overlap:")
for pair, n in overlap.items():
    print(f"  {pair:16} {n}")

dd.assert_no_cross_split_duplicates(joined)
print("\nContent-level disjointness: PASSED")

Cross-split content-hash overlap:
  train&val        30
  train&test       38
  val&test         26


AssertionError: content hash leaks across splits: {'train&val': 30, 'train&test': 38, 'val&test': 26}

In [17]:
train_hashes = set(joined.loc[joined["split"].astype(str) == "train", "sha256"])
val_hashes   = set(joined.loc[joined["split"].astype(str) == "val",   "sha256"])
test_hashes  = set(joined.loc[joined["split"].astype(str) == "test",  "sha256"])

colliding = (train_hashes & val_hashes) | (train_hashes & test_hashes) | (val_hashes & test_hashes)
print(f"Distinct colliding hashes: {len(colliding)}")

collision_rows = joined[joined["sha256"].isin(colliding)].copy()
collision_rows["content_len"] = collision_rows["bytes"] if "bytes" in collision_rows else None

# Pull actual file content for the colliding hashes to see what they are
for h in list(colliding)[:15]:
    rows = collision_rows[collision_rows["sha256"] == h]
    sample_path = SOURCES / rows.iloc[0]["rel_path"]
    content = sample_path.read_bytes()
    print(f"\nhash={h[:12]}...  n_files={len(rows)}  splits={sorted(rows['split'].astype(str).unique())}")
    print(f"  content ({len(content)} bytes): {content[:80]!r}")

Distinct colliding hashes: 66

hash=239d0b466e9e...  n_files=3  splits=['test', 'train']
  content (139 bytes): b'#include<iostream>\nusing namespace std;\nint main()\n{\n  int H,W;\n  for(int i = 0;'

hash=5feceb66ffc8...  n_files=2  splits=['test', 'train']
  content (1 bytes): b'0'

hash=ebd5691dccbd...  n_files=2  splits=['train', 'val']
  content (658 bytes): b'#include<bits/stdc++.h>\nusing namespace std;\nint main()\n{\n    double x,y,z,p;\n  '

hash=74234e98afe7...  n_files=6  splits=['train', 'val']
  content (4 bytes): b'null'

hash=25961242a955...  n_files=2  splits=['train', 'val']
  content (455 bytes): b'#include<bits/stdc++.h>\nusing namespace std;\n\nint main () {\n  int N;\n  vector<in'

hash=90027a2fd3c4...  n_files=2  splits=['test', 'train']
  content (343 bytes): b'#include<bits/stdc++.h>\n\n#define int long long\n\nusing namespace std;\n\nint main()'

hash=8622c3f33c18...  n_files=3  splits=['test', 'train', 'val']
  content (24 bytes): b'\xe3\x81\x8a\xe5\x89\x8d\

In [18]:
label_consistency = (
    joined[joined["sha256"].isin(colliding)]
    .groupby("sha256")["coarse_label"]
    .agg(lambda x: sorted(x.astype(str).unique()))
)

n_consistent = (label_consistency.apply(len) == 1).sum()
n_ambiguous = (label_consistency.apply(len) > 1).sum()

print(f"Colliding hashes with ONE label everywhere:      {n_consistent}")
print(f"Colliding hashes with MULTIPLE labels:            {n_ambiguous}")

if n_ambiguous:
    print("\nAmbiguous examples (same code, different verdicts):")
    print(label_consistency[label_consistency.apply(len) > 1].head(10).to_string())

Colliding hashes with ONE label everywhere:      56
Colliding hashes with MULTIPLE labels:            10

Ambiguous examples (same code, different verdicts):
sha256
1fa918d2357561552d74978e5dd5f2c4311b50581762c2df01a031e75c3c495b          [LOGICAL, RUNTIME_ERROR]
32b7a5c886ff75a55d7389736d44f5342c4b0b1bab861dddfb38acc8665957cd    [COMPILE_ERROR, RUNTIME_ERROR]
3748533b79aaaf135ab6a47697fc4a83ee0a3a4634b96ce1fa259054855812bf          [LOGICAL, RUNTIME_ERROR]
64662310b5925d0601a4e3acbe371a178101a26d2c1aa2fc3eb55e6420b53877          [LOGICAL, RUNTIME_ERROR]
74234e98afe7498fb5daf1f36ac2d78acc339464f950703b8c019892f982b90b    [COMPILE_ERROR, RUNTIME_ERROR]
809ce5e98bccb0868e724e2a3bcd12eb76bfe0e0c7015e2bfd8a44af802af6e4          [LOGICAL, RUNTIME_ERROR]
8caba2b5256a982c15a4606957e38addae8cb142e39ec67811db5d23e5c739e2       [ERROR_FREE, RUNTIME_ERROR]
9acc86c426f0317dcffc0d8819f48eaf54c013e2900111f12733dedf06dfa3ee          [COMPILE_ERROR, LOGICAL]
bddb37e6d0ea65681a232cfd1463401841c4b59a89e

In [19]:
colliding_rows = joined[joined["sha256"].isin(colliding)]
print(f"Total sampled rows touched by cross-split hash collisions: {len(colliding_rows)}")
print(f"As % of the 75,000-file corpus: {len(colliding_rows)/750:.3f}%")

# For each colliding hash, keep only the split holding the plurality of its rows
keep_split = (
    colliding_rows.groupby("sha256")["split"]
    .agg(lambda s: s.astype(str).value_counts().idxmax())
)

joined["_keep_split"] = joined["sha256"].map(keep_split)
drop_mask = (
    joined["sha256"].isin(colliding)
    & (joined["split"].astype(str) != joined["_keep_split"])
)
print(f"Rows to drop: {drop_mask.sum()}")

sample_clean = joined.loc[~drop_mask].drop(columns=["_keep_split"]).copy()
print(f"\nCorpus size after dedup exclusion: {len(sample_clean):,} (was 75,000)")

print("\nQuota impact by (split, class):")
print(pd.crosstab(sample_clean["split"], sample_clean["coarse_label"]).to_string())

Total sampled rows touched by cross-split hash collisions: 326
As % of the 75,000-file corpus: 0.435%
Rows to drop: 150

Corpus size after dedup exclusion: 74,850 (was 75,000)

Quota impact by (split, class):
coarse_label  ERROR_FREE  COMPILE_ERROR  RUNTIME_ERROR  LOGICAL
split                                                          
train               5999          17987          14994     5998
val                 2000           5951           4993     1997
test                2000           5938           4994     1999


In [20]:
overlap_clean = dd.cross_split_duplicate_hashes(sample_clean)
print("Cross-split content-hash overlap after exclusion:")
for pair, n in overlap_clean.items():
    print(f"  {pair:16} {n}")

dd.assert_no_cross_split_duplicates(sample_clean)
print("\nContent-level disjointness: PASSED")

Cross-split content-hash overlap after exclusion:
  train&val        0
  train&test       0
  val&test         0

Content-level disjointness: PASSED


In [21]:
sample_final = sample_clean.merge(
    hashes_indexed[["sha256", "bytes"]].reset_index().rename(columns={"path": "rel_path"}),
    on=["rel_path", "sha256"],
    how="left",
    suffixes=("", "_h"),
)

DEDUP_OUT = ROOT / "data" / "processed" / "splits" / "sample_manifest_hashed.parquet"
sample_final.to_parquet(DEDUP_OUT, compression="zstd", index=False)

report_final = dd.dedup_report(hashes)  # unchanged — describes the raw 75k hash pass
report_final["rows_dropped_cross_split_dedup"] = int(75_000 - len(sample_final))
pd.Series(report_final).to_csv(REPORTS / "dedup_report.csv", header=["value"])

print(f"{DEDUP_OUT.name}: {DEDUP_OUT.stat().st_size/1e6:.2f} MB, {len(sample_final):,} rows")
print("dedup_report.csv written")

sample_manifest_hashed.parquet: 5.48 MB, 74,850 rows
dedup_report.csv written


In [22]:
import json
import hashlib
from datetime import datetime, timezone

def file_sha256(path: Path) -> str:
    """SHA-256 of an entire file, in chunks so we don't load large files whole."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

FINAL_MANIFEST = ROOT / "data" / "processed" / "splits" / "sample_manifest_hashed.parquet"

config = {
    "milestone": "M2",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),

    "taxonomy_version": tax.TAXONOMY_VERSION,

    "split": {
        "method": "problem_level_split",
        "seed": sp.DEFAULT_SEED,
        "ratios": {k.value: v for k, v in sp.DEFAULT_RATIOS.items()},
    },
    "comparison_split": {
        "method": "submission_level_split",
        "seed": sp.DEFAULT_SEED,
        "note": "deliberately unsafe; used only to measure the leakage effect",
    },

    "sampling": {
        "method": "draw_sample (round-robin per problem)",
        "seed": smp.DEFAULT_SEED,
        "class_totals": {k.value: v for k, v in smp.DEFAULT_CLASS_TOTALS.items()},
    },

    "dedup": {
        "method": "exact SHA-256 on normalized bytes (CRLF->LF, BOM stripped, trailing whitespace stripped)",
        "raw_sample_size": 75_000,
        "cross_split_collisions_found": 66,
        "rows_dropped": 150,
        "final_size": len(sample_clean),
    },

    "row_counts": {
        "raw_metadata": 8_008_527,
        "candidate_pool": 7_640_056,
        "discarded": 368_471,
        "working_corpus_final": len(sample_clean),
    },

    "findings": {
        "chains_straddling_random_split": 979_741,
        "chains_straddling_problem_split": 0,
        "linker_pilot_rate_pct": 1.24,
        "linker_pilot_n_sampled": 500,
        "linker_pilot_compiler": "g++ (MinGW.org GCC-6.3.0-1) 6.3.0",
        "linker_pilot_note": "lower bound; re-measure with modern GCC at Tier 2",
    },

    "artifact_hashes": {
        "sample_manifest_hashed.parquet": file_sha256(FINAL_MANIFEST),
    },

    "environment_note": (
        "Windows Defender real-time scanning must be excluded from "
        "data/ and the CodeNet archive directory, or per-file hashing "
        "and extraction throughput degrades by ~40x after the on-access "
        "scan cache fills (observed: 2,780 files/sec -> 70 files/sec)."
    ),
}

CONFIG_OUT = ROOT / "data" / "processed" / "splits" / "split_config.json"
CONFIG_OUT.write_text(json.dumps(config, indent=2), encoding="utf-8")
print(f"Written: {CONFIG_OUT}")
print(json.dumps(config, indent=2))

Written: d:\Dev\Github\transformer-defect-prediction\data\processed\splits\split_config.json
{
  "milestone": "M2",
  "generated_at_utc": "2026-08-08T19:22:17.784822+00:00",
  "taxonomy_version": "1.0",
  "split": {
    "method": "problem_level_split",
    "seed": 42,
    "ratios": {
      "train": 0.6,
      "val": 0.2,
      "test": 0.2
    }
  },
  "comparison_split": {
    "method": "submission_level_split",
    "seed": 42,
    "note": "deliberately unsafe; used only to measure the leakage effect"
  },
  "sampling": {
    "method": "draw_sample (round-robin per problem)",
    "seed": 42,
    "class_totals": {
      "ERROR_FREE": 10000,
      "COMPILE_ERROR": 30000,
      "RUNTIME_ERROR": 25000,
      "LOGICAL": 10000
    }
  },
  "dedup": {
    "method": "exact SHA-256 on normalized bytes (CRLF->LF, BOM stripped, trailing whitespace stripped)",
    "raw_sample_size": 75000,
    "cross_split_collisions_found": 66,
    "rows_dropped": 150,
    "final_size": 74850
  },
  "row_counts":

In [9]:
chains = df.groupby(["user_id", "problem_id"], observed=True).agg(
    n=("submission_id", "size"),
    n_problem_split=("split", "nunique"),
    n_random_split=("random_split", "nunique"),
)
multi = chains["n"] > 1

print(f"(user, problem) chains:            {len(chains):,}")
print(f"  with more than one submission:   {multi.sum():,}")
print(f"  submissions inside those chains: {chains.loc[multi, 'n'].sum():,}")
print()
print(f"Chains straddling splits — problem-level: {(chains.loc[multi, 'n_problem_split'] > 1).sum():,}")
print(f"Chains straddling splits — random:        {(chains.loc[multi, 'n_random_split'] > 1).sum():,}")

(user, problem) chains:            3,992,111
  with more than one submission:   1,364,569
  submissions inside those chains: 5,012,514

Chains straddling splits — problem-level: 0
Chains straddling splits — random:        979,741
